In [1]:
# Solar Flare Classification using Support Vector Machines
# =======================================================
#
# In this tutorial, we'll build an SVM model to predict solar flare classifications
# using data from NOAA's Space Weather Prediction Center.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
import urllib.request
import io
import zipfile

# Set random seed for reproducibility
np.random.seed(42)

url = "https://github.com/khanhnamle1994/MetaRec/raw/master/Datasets-and-Benchmarks/solar_flare.csv"

response = urllib.request.urlopen(url)
flare_data = pd.read_csv(io.StringIO(response.read().decode('utf-8')))
print(f"Dataset shape: {flare_data.shape}")

HTTPError: HTTP Error 404: Not Found

In [ ]:
# Display the first few rows
print("\nFirst few rows of the dataset:")
print(flare_data.head())

# Data exploration
# ---------------
print("\nData exploration:")
print(flare_data.info())
print("\nSummary statistics:")
print(flare_data.describe())

# Check class distribution
print("\nClass distribution:")
class_counts = flare_data['class'].value_counts()
print(class_counts)

# Plot class distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='class', data=flare_data)
plt.title('Solar Flare Class Distribution')
plt.xlabel('Flare Class')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Data preprocessing
# -----------------
print("\nPreprocessing the data...")

# Convert categorical features to numeric
# The dataset has the following features:
# 1. Code for class (modified Zurich class)
# 2. Code for largest spot size
# 3. Code for spot distribution
# 4. Activity
# 5. Evolution
# 6. Previous 24-hour flare activity code
# 7. Historically-complex
# 8. Did region become historically complex on this pass?
# 9. Area
# 10. Area of the largest spot
# 11-13. Target variables: C-class, M-class, X-class flares

# Define features and target
X = flare_data.drop(['class', 'C-class', 'M-class', 'X-class'], axis=1)
# We'll predict the 'class' column (combined classification)
y = flare_data['class']

# Encode categorical target
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Let's see the mapping
print("\nClass encoding mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label} -> {i}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, 
                                                   test_size=0.25, 
                                                   random_state=42, 
                                                   stratify=y_encoded)

print(f"\nTraining set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

# Building the SVM model
# ---------------------
print("\nBuilding and tuning the SVM model...")

# Create a pipeline with preprocessing and model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=42))
])

# Define parameter grid for GridSearchCV
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.1, 0.01],
    'svm__kernel': ['rbf', 'poly', 'sigmoid']
}

# Set up GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)

# Fit the model
grid_search.fit(X_train, y_train)

# Print best parameters
print("\nBest parameters found:")
print(grid_search.best_params_)

# Evaluating the model
# -------------------
print("\nEvaluating the model...")

# Get the best model
best_model = grid_search.best_estimator_

# Make predictions
y_pred = best_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Generate classification report
print("\nClassification Report:")
class_names = label_encoder.classes_
print(classification_report(y_test, y_pred, target_names=class_names))

# Plot confusion matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.tight_layout()
plt.show()

# Feature importance analysis
# --------------------------
print("\nAnalyzing feature importance...")

# For SVM, we can analyze feature importance indirectly
# We'll use a model with a linear kernel to extract coefficients
linear_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', random_state=42))
])

linear_svm.fit(X_train, y_train)

# Extract feature coefficients
feature_importance = np.abs(linear_svm['svm'].coef_)
# Average importance across classes (for multi-class)
avg_importance = np.mean(feature_importance, axis=0)

# Create feature importance dataframe
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': avg_importance
})
importance_df = importance_df.sort_values('Importance', ascending=False)

# Plot feature importance
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Feature Importance (based on Linear SVM coefficients)')
plt.tight_layout()
plt.show()

# Visualizing the decision boundaries (for 2 most important features)
# -----------------------------------------------------------------
if importance_df.shape[0] >= 2:
    print("\nVisualizing decision boundaries using the 2 most important features...")
    
    # Get the two most important features
    top_features = importance_df['Feature'].iloc[:2].tolist()
    
    # Create a new model using only these features
    X_train_2d = X_train[top_features]
    X_test_2d = X_test[top_features]
    
    # Train a new model
    model_2d = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(probability=True, kernel=grid_search.best_params_['svm__kernel'],
                  C=grid_search.best_params_['svm__C'], 
                  gamma=grid_search.best_params_['svm__gamma'],
                  random_state=42))
    ])
    model_2d.fit(X_train_2d, y_train)
    
    # Create a mesh grid
    x_min, x_max = X_train_2d.iloc[:, 0].min() - 1, X_train_2d.iloc[:, 0].max() + 1
    y_min, y_max = X_train_2d.iloc[:, 1].min() - 1, X_train_2d.iloc[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                         np.arange(y_min, y_max, 0.1))
    
    # Make predictions on the mesh grid
    Z = model_2d.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot the decision boundary
    plt.figure(figsize=(12, 10))
    plt.contourf(xx, yy, Z, alpha=0.4)
    scatter = plt.scatter(X_test_2d.iloc[:, 0], X_test_2d.iloc[:, 1], c=y_test, 
               edgecolors='k', alpha=0.8, cmap=plt.cm.get_cmap('viridis', len(class_names)))
    plt.xlabel(top_features[0])
    plt.ylabel(top_features[1])
    plt.title(f'Decision Boundaries using {top_features[0]} and {top_features[1]}')
    plt.colorbar(scatter, ticks=range(len(class_names)))
    plt.tight_layout()
    plt.show()

# Saving the model
# ---------------
import joblib

print("\nSaving the model...")
joblib.dump(best_model, 'solar_flare_svm_model.pkl')
print("Model saved as 'solar_flare_svm_model.pkl'")

print("\nTutorial complete!")